<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025_TASK2/blob/main/ORJ_DCASE_EVALUATION_FILES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# SELECTED-FEATURE kNN EUCLIDEAN OUTPUT GENERATOR
# DCASE 2025 Task 2
#
# Uses:
#   - Fixed selected features per machine type
#   - Fixed score direction per machine type
#   - StandardScaler + kNN Euclidean distance, k=10
#
# Creates:
#   anomaly_score_{MachineType}_section_00_test.csv
#   decision_result_{MachineType}_section_00_test.csv
#
# Creates downloadable ZIP:
#   /content/dcase2025_selected_feature_knn_submission.zip
# ============================================================

import os
import glob
import json
import shutil
import warnings
import subprocess
import sys

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

warnings.filterwarnings("ignore")

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpyxl"])


# ============================================================
# 1. CONFIGURATION
# ============================================================

BASE_DIR = "/content"


# It is not required for creating the output CSV files.
EVAL_DIR = "/content/dcase2025_task2_evaluator"

OUTPUT_DIR = os.path.join(BASE_DIR, "selected_feature_knn_outputs")
SUBMISSION_ROOT = os.path.join(BASE_DIR, "dcase2025_selected_feature_knn_submission")

TEAM_NAME = "METU"
SYSTEM_NAME = "TDA"

TEAM_DIR = os.path.join(SUBMISSION_ROOT, "teams", TEAM_NAME, SYSTEM_NAME)

KNN_K = 10
P_AUC_FPR = 0.1

# If True, missing selected features cause an error.
# If False, the code uses only available selected features.
STRICT_FEATURE_CHECK = True

MACHINE_TYPES = [
    "AutoTrash",
    "BandSealer",
    "CoffeeGrinder",
    "HomeCamera",
    "Polisher",
    "ScrewFeeder",
    "ToyPet",
    "ToyRCCar",
]


# ============================================================
# 2. SELECTED FEATURES PER MACHINE TYPE
# ============================================================

best_features_by_machine = {
    "AutoTrash": [
        "H0_max_lifetime",
        "H0_lifetime_skew",
        "H0_l2_norm_lifetime",
        "H0_lifetime_energy",
        "H0_max_diag_distance",
        "H0_betti_argmax",
        "H0_landscape_layer2_max",
        "H0_landscape_layer2_mean",
        "H0_landscape_layer3_mean",
        "H0_landscape_layer4_max",
        "H0_landscape_layer4_mean",
        "H0_landscape_layer5_auc",
        "H0_silhouette_auc",
        "H0_pimage_std",
        "H0_pimage_max",
        "H0_weighted_death_std",
        "H1_tail_share_q90",
        "H1_top1_share",
        "H1_top5_share",
        "H1_birth_max",
        "H1_birth_q75",
        "H1_death_max",
        "H1_mean_birth_death_ratio",
        "H1_betti_max",
        "H1_silhouette_max",
    ],

    "BandSealer": [
        "H0_range_lifetime",
        "H0_tail_share_q95",
        "H0_top3_share",
        "H0_birth_min",
        "H0_death_kurtosis",
        "H0_pimage_min",
        "H1_tail_share_q95",
        "H1_birth_q25",
        "H1_weighted_death_mean",
        "H1_weighted_midlife_mean",
        "H1_to_H0_max_lifetime_ratio",
        "H1_to_H0_num_points_ratio",
        "H1_to_H0_entropy_ratio",
    ],

    "CoffeeGrinder": [
        "H0_q95_lifetime",
        "H0_iqr_lifetime",
        "H0_normalized_persistence_entropy",
        "H0_birth_kurtosis",
        "H0_betti_entropy",
        "H1_min_lifetime",
        "H1_q10_lifetime",
        "H1_top1_share",
        "H1_birth_kurtosis",
        "H1_death_kurtosis",
        "H1_std_midlife",
        "H1_min_midlife",
        "H1_betti_std",
        "H1_betti_entropy",
        "H1_betti_num_peaks",
        "H1_landscape_std",
        "H1_landscape_layer1_auc",
        "H1_landscape_layer5_mean",
        "H1_silhouette_var",
        "H1_silhouette_l2",
        "H1_pimage_std",
        "H1_pimage_var",
        "H1_pimage_max",
        "H1_pimage_entropy",
        "H1_weighted_death_std",
    ],

    "HomeCamera": [
        "H0_sum_lifetime",
        "H0_mean_lifetime",
        "H0_q10_lifetime",
        "H0_q25_lifetime",
        "H0_iqr_lifetime",
        "H0_birth_min",
        "H0_death_min",
        "H0_death_median",
        "H0_mean_diag_distance",
        "H0_pimage_l1",
        "H0_Carlsson_f3",
        "H1_sum_lifetime",
        "H1_q10_lifetime",
        "H1_l1_norm_lifetime",
        "H1_birth_median",
        "H1_death_median",
        "H1_sum_diag_distance",
        "H1_betti_mean",
        "H1_betti_l1",
        "H1_landscape_l1",
        "H1_landscape_layer1_auc",
        "H1_to_H0_num_points_ratio",
    ],

    "Polisher": [
        "H0_mean_lifetime",
        "H0_q10_lifetime",
        "H0_birth_max",
        "H0_betti_num_peaks",
        "H0_landscape_mean",
        "H0_landscape_layer5_max",
        "H0_pimage_min",
        "H1_sum_lifetime",
        "H1_entropy_lifetime_ratio",
        "H1_l2_norm_lifetime",
        "H1_lifetime_energy",
        "H1_lifetime_rms",
        "H1_tail_share_q80",
        "H1_tail_share_q90",
        "H1_birth_std",
        "H1_landscape_layer2_auc",
        "H1_landscape_layer2_max",
        "H1_landscape_layer3_max",
        "H1_landscape_layer5_max",
        "H1_silhouette_std",
        "H1_silhouette_l2",
    ],

    "ScrewFeeder": [
        "H1_points_raw",
        "H0_top1_share",
        "H0_top5_share",
        "H0_birth_skew",
        "H0_birth_kurtosis",
        "H0_landscape_entropy",
        "H0_landscape_layer4_max",
        "H0_silhouette_entropy",
        "H0_pimage_std",
        "H1_q95_lifetime",
        "H1_birth_skew",
        "H1_birth_kurtosis",
        "H1_death_std",
        "H1_death_skew",
        "H1_death_kurtosis",
        "H1_std_midlife",
        "H1_betti_var",
        "H1_betti_num_peaks",
        "H1_landscape_layer2_max",
        "H1_silhouette_std",
        "H1_silhouette_var",
        "H1_silhouette_max",
        "H1_to_H0_num_points_ratio",
        "H1_to_H0_energy_ratio",
    ],

    "ToyPet": [
        "H0_entropy_lifetime_ratio",
        "H0_death_min",
        "H0_betti_mean",
        "H0_betti_l1",
        "H0_landscape_layer3_max",
        "H0_pimage_min",
        "H0_Carlsson_f3",
        "H1_num_points",
        "H1_q10_lifetime",
        "H1_q95_lifetime",
        "H1_persistence_entropy",
        "H1_linf_norm_lifetime",
        "H1_top1_share",
        "H1_top5_share",
        "H1_landscape_entropy",
        "H1_landscape_layer3_max",
        "H1_to_H0_energy_ratio",
    ],

    "ToyRCCar": [
        "H0_num_points",
        "H0_persistence_entropy",
        "H0_top3_share",
        "H0_landscape_var",
        "H0_landscape_layer1_max",
        "H0_landscape_layer3_max",
        "H0_landscape_layer4_max",
        "H0_landscape_layer5_max",
        "H0_pimage_entropy",
        "H0_Carlsson_f3",
        "H1_q90_lifetime",
        "H1_iqr_lifetime",
        "H1_range_lifetime",
        "H1_persistence_entropy",
        "H1_death_max",
        "H1_betti_l2",
        "H1_betti_energy",
        "H1_landscape_layer2_max",
        "H1_silhouette_entropy",
        "H1_pimage_energy",
        "H1_pimage_l2",
        "H1_to_H0_sum_lifetime_ratio",
        "H1_to_H0_num_points_ratio",
        "H1_to_H0_entropy_ratio",
    ],
}


# ============================================================
# 3. FIXED SCORE DIRECTIONS PER MACHINE
# ============================================================

SCORE_DIRECTION_BY_MACHINE = {
    "AutoTrash": "higher_distance_more_anomaly",
    "BandSealer": "lower_distance_more_anomaly",
    "CoffeeGrinder": "lower_distance_more_anomaly",
    "HomeCamera": "higher_distance_more_anomaly",
    "Polisher": "lower_distance_more_anomaly",
    "ScrewFeeder": "higher_distance_more_anomaly",
    "ToyPet": "higher_distance_more_anomaly",
    "ToyRCCar": "higher_distance_more_anomaly",
}


def apply_fixed_score_direction(machine, raw_distance_scores):
    """
    Converts raw kNN distance into final anomaly score.

    Final rule:
        larger final_score = more anomalous

    higher_distance_more_anomaly:
        final_score = raw kNN distance

    lower_distance_more_anomaly:
        final_score = -raw kNN distance
    """

    if machine not in SCORE_DIRECTION_BY_MACHINE:
        raise KeyError(f"No score direction defined for machine: {machine}")

    direction = SCORE_DIRECTION_BY_MACHINE[machine]

    raw_distance_scores = np.asarray(raw_distance_scores, dtype=float)

    if direction == "higher_distance_more_anomaly":
        final_scores = raw_distance_scores

    elif direction == "lower_distance_more_anomaly":
        final_scores = -raw_distance_scores

    else:
        raise ValueError(
            f"Invalid score direction for {machine}: {direction}"
        )

    return direction, final_scores


# ============================================================
# 4. CLEAN AND CREATE OUTPUT FOLDERS
# ============================================================

for folder in [OUTPUT_DIR, SUBMISSION_ROOT]:
    if os.path.exists(folder):
        shutil.rmtree(folder)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEAM_DIR, exist_ok=True)

print("Output folder:")
print(OUTPUT_DIR)

print("\nSubmission team folder:")
print(TEAM_DIR)


# ============================================================
# 5. FILE HELPERS
# ============================================================

def latest_file(patterns):
    files = []
    for pattern in patterns:
        files.extend(glob.glob(pattern, recursive=True))
    files = sorted(set(files), key=os.path.getmtime, reverse=True)
    return files[0] if files else None


def find_feature_file(machine):
    patterns = [
        os.path.join(BASE_DIR, f"cubical_mel_tda_features_{machine}*.xlsx"),
        os.path.join(BASE_DIR, "**", f"cubical_mel_tda_features_{machine}*.xlsx"),
    ]

    return latest_file(patterns)


def find_label_ground_truth_file(machine):
    official_path = os.path.join(
        EVAL_DIR,
        "ground_truth_data",
        f"ground_truth_{machine}_section_00_test.csv"
    )

    if os.path.exists(official_path):
        return official_path

    patterns = [
        os.path.join(BASE_DIR, f"ground_truth_{machine}_section_00_test*.csv"),
        os.path.join(BASE_DIR, "**", f"ground_truth_{machine}_section_00_test*.csv"),
    ]

    return latest_file(patterns)


def find_domain_ground_truth_file(machine):
    official_path = os.path.join(
        EVAL_DIR,
        "ground_truth_domain",
        f"ground_truth_{machine}_section_00_test.csv"
    )

    if os.path.exists(official_path):
        return official_path

    patterns = [
        os.path.join(BASE_DIR, f"ground_truth_domain_{machine}_section_00_test*.csv"),
        os.path.join(BASE_DIR, f"ground_truth_*domain*{machine}*.csv"),
        os.path.join(BASE_DIR, "**", f"ground_truth_domain_{machine}_section_00_test*.csv"),
        os.path.join(BASE_DIR, "**", f"ground_truth_*domain*{machine}*.csv"),
    ]

    return latest_file(patterns)


def read_two_col_csv(path, second_name):
    df = pd.read_csv(path, header=None)
    df = df.iloc[:, :2].copy()
    df.columns = ["file_id", second_name]
    df["file_id"] = df["file_id"].astype(str)

    return df


def read_label_ground_truth(path):
    gt = read_two_col_csv(path, "y_true")

    label_map = {
        "normal": 0,
        "anomaly": 1,
        "negative": 0,
        "positive": 1,
        "0": 0,
        "1": 1,
    }

    gt["y_true"] = gt["y_true"].astype(str).str.strip().str.lower().map(
        lambda x: label_map.get(x, x)
    )
    gt["y_true"] = gt["y_true"].astype(int)

    return gt


def read_domain_ground_truth(path):
    dom = read_two_col_csv(path, "domain")

    domain_map = {
        "source": "source",
        "target": "target",
        "0": "source",
        "1": "target",
    }

    dom["domain"] = dom["domain"].astype(str).str.strip().str.lower().map(
        lambda x: domain_map.get(x, x)
    )

    return dom


# ============================================================
# 6. DCASE-LIKE METRIC HELPERS
# ============================================================

def dcase_heaviside_auc(pos_scores, neg_scores):
    pos_scores = np.asarray(pos_scores, dtype=float)
    neg_scores = np.asarray(neg_scores, dtype=float)

    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return np.nan

    comp = pos_scores[:, None] - neg_scores[None, :]
    return np.mean(comp > 0)


def harmonic_mean(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    if np.any(values <= 0):
        return 0.0

    return len(values) / np.sum(1.0 / values)


def dcase_auc_source_target_pauc(y_true, scores, domains, p=0.1):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores).astype(float)
    domains = np.asarray(domains).astype(str)

    anomaly_scores = scores[y_true == 1]
    normal_scores_all = scores[y_true == 0]

    source_normal_scores = scores[(y_true == 0) & (domains == "source")]
    target_normal_scores = scores[(y_true == 0) & (domains == "target")]

    auc_source = dcase_heaviside_auc(anomaly_scores, source_normal_scores)
    auc_target = dcase_heaviside_auc(anomaly_scores, target_normal_scores)

    n_normals = len(normal_scores_all)
    top_k = max(1, int(np.floor(p * n_normals)))

    hardest_normal_scores = np.sort(normal_scores_all)[::-1][:top_k]

    pauc = dcase_heaviside_auc(anomaly_scores, hardest_normal_scores)

    dcase_hmean = harmonic_mean([auc_source, auc_target, pauc])

    return {
        "auc_source": auc_source,
        "auc_target": auc_target,
        "pauc": pauc,
        "dcase_hmean": dcase_hmean,
        "auc_mean_source_target": np.nanmean([auc_source, auc_target]),
    }


# ============================================================
# 7. kNN DISTANCE SCORING
# ============================================================

def knn_distance_scores(X, k=10):
    n_samples = X.shape[0]
    k_eff = min(k, n_samples - 1)

    if k_eff < 1:
        raise ValueError("Need at least 2 samples for kNN scoring.")

    nn = NearestNeighbors(
        n_neighbors=k_eff + 1,
        metric="euclidean",
    )

    nn.fit(X)

    distances, _ = nn.kneighbors(X)

    distances_no_self = distances[:, 1:k_eff + 1]

    return distances_no_self.mean(axis=1)


# ============================================================
# 8. MAIN MACHINE PROCESSOR
# ============================================================

def process_machine(machine):
    print("\n" + "=" * 100)
    print(f"PROCESSING MACHINE: {machine}")
    print("=" * 100)

    feature_file = find_feature_file(machine)

    if feature_file is None:
        raise FileNotFoundError(f"No feature matrix found for {machine}")

    print("Feature file:", feature_file)

    df = pd.read_excel(feature_file)

    if "file_id" not in df.columns:
        raise ValueError(f"'file_id' column not found in feature matrix for {machine}")

    df["file_id"] = df["file_id"].astype(str)

    selected_features = best_features_by_machine[machine]

    missing_features = [f for f in selected_features if f not in df.columns]
    available_features = [f for f in selected_features if f in df.columns]

    print(f"Selected features requested : {len(selected_features)}")
    print(f"Available selected features : {len(available_features)}")
    print(f"Missing selected features   : {len(missing_features)}")

    if missing_features:
        print("\nMissing features:")
        for f in missing_features:
            print(" -", f)

        if STRICT_FEATURE_CHECK:
            raise ValueError(
                f"{machine}: selected features are missing. "
                "Set STRICT_FEATURE_CHECK=False to continue with available features."
            )

    if len(available_features) == 0:
        raise ValueError(f"{machine}: no selected features are available.")

    X_df = df[available_features].copy()

    X_df = X_df.apply(pd.to_numeric, errors="coerce")
    X_df = X_df.replace([np.inf, -np.inf], np.nan)
    X_df = X_df.fillna(X_df.median(numeric_only=True))
    X_df = X_df.fillna(0.0)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_df.values.astype(float))

    raw_distance = knn_distance_scores(X_scaled, k=KNN_K)

    direction, final_scores = apply_fixed_score_direction(
        machine=machine,
        raw_distance_scores=raw_distance,
    )

    print("Score direction from dictionary:", direction)

    # --------------------------------------------------------
    # Optional ground-truth loading for report only
    # --------------------------------------------------------

    label_file = find_label_ground_truth_file(machine)
    domain_file = find_domain_ground_truth_file(machine)

    y_true = None
    domains = None
    gt_available = False
    direction_metrics = {}

    if label_file is not None and domain_file is not None:
        try:
            gt = read_label_ground_truth(label_file)
            dom = read_domain_ground_truth(domain_file)

            meta = df[["file_id"]].copy()
            meta = meta.merge(gt, on="file_id", how="left")
            meta = meta.merge(dom, on="file_id", how="left")

            if meta["y_true"].notna().all() and meta["domain"].notna().all():
                y_true = meta["y_true"].astype(int).values
                domains = meta["domain"].astype(str).values
                gt_available = True

                print("GT available: YES")
                print("Label GT:", label_file)
                print("Domain GT:", domain_file)

                direction_metrics = dcase_auc_source_target_pauc(
                    y_true=y_true,
                    scores=final_scores,
                    domains=domains,
                    p=P_AUC_FPR,
                )

                print(
                    "DCASE-like metrics with fixed direction | "
                    f"AUC_source={direction_metrics['auc_source']:.5f} | "
                    f"AUC_target={direction_metrics['auc_target']:.5f} | "
                    f"pAUC={direction_metrics['pauc']:.5f} | "
                    f"hmean={direction_metrics['dcase_hmean']:.5f}"
                )

            else:
                print("GT available: PARTIAL. Metrics will not be reported.")

        except Exception as e:
            print("GT loading failed. Metrics will not be reported.")
            print(str(e))

    else:
        print("GT available: NO. Metrics will not be reported.")


    if y_true is not None:
        n_anomaly = int((y_true == 1).sum())
    else:
        n_anomaly = len(df) // 2

    sorted_idx = np.argsort(final_scores)[::-1]

    decisions = np.zeros(len(df), dtype=int)
    decisions[sorted_idx[:n_anomaly]] = 1

    decision_metrics = {}

    if y_true is not None:
        tn, fp, fn, tp = confusion_matrix(y_true, decisions, labels=[0, 1]).ravel()

        decision_metrics = {
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp),
            "accuracy": accuracy_score(y_true, decisions),
            "precision": precision_score(y_true, decisions, zero_division=0),
            "recall": recall_score(y_true, decisions, zero_division=0),
            "f1": f1_score(y_true, decisions, zero_division=0),
        }

        print("Decision metrics, not used for official AUC:")
        print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
        print(
            f"Accuracy={decision_metrics['accuracy']:.5f} | "
            f"Precision={decision_metrics['precision']:.5f} | "
            f"Recall={decision_metrics['recall']:.5f} | "
            f"F1={decision_metrics['f1']:.5f}"
        )

    # --------------------------------------------------------
    # Save output files
    # --------------------------------------------------------

    machine_out_dir = os.path.join(OUTPUT_DIR, machine)
    os.makedirs(machine_out_dir, exist_ok=True)

    score_file = os.path.join(
        machine_out_dir,
        f"anomaly_score_{machine}_section_00_test.csv"
    )

    decision_file = os.path.join(
        machine_out_dir,
        f"decision_result_{machine}_section_00_test.csv"
    )

    score_df = pd.DataFrame({
        0: df["file_id"].astype(str).values,
        1: final_scores,
    })

    decision_df = pd.DataFrame({
        0: df["file_id"].astype(str).values,
        1: decisions.astype(int),
    })

    score_df.to_csv(score_file, index=False, header=False)
    decision_df.to_csv(decision_file, index=False, header=False)

    shutil.copy2(score_file, os.path.join(TEAM_DIR, os.path.basename(score_file)))
    shutil.copy2(decision_file, os.path.join(TEAM_DIR, os.path.basename(decision_file)))

    print("Saved score file   :", score_file)
    print("Saved decision file:", decision_file)

    result = {
        "machine": machine,
        "feature_file": feature_file,
        "n_samples": len(df),
        "n_selected_features_requested": len(selected_features),
        "n_selected_features_used": len(available_features),
        "n_missing_features": len(missing_features),
        "score_direction": direction,
        "score_min": float(np.min(final_scores)),
        "score_max": float(np.max(final_scores)),
        "score_mean": float(np.mean(final_scores)),
        "decision_0_count": int((decisions == 0).sum()),
        "decision_1_count": int((decisions == 1).sum()),
    }

    if direction_metrics:
        result.update({
            "auc_source": direction_metrics["auc_source"],
            "auc_target": direction_metrics["auc_target"],
            "pauc": direction_metrics["pauc"],
            "dcase_hmean": direction_metrics["dcase_hmean"],
            "auc_mean_source_target": direction_metrics["auc_mean_source_target"],
        })

    if decision_metrics:
        result.update(decision_metrics)

    return result


# ============================================================
# 9. RUN ALL MACHINES
# ============================================================

all_results = []

for machine in MACHINE_TYPES:
    try:
        result = process_machine(machine)
        all_results.append(result)
        print(f"\nFINISHED {machine}\n")

    except Exception as e:
        print("\n" + "!" * 100)
        print(f"ERROR IN MACHINE: {machine}")
        print(str(e))
        print("!" * 100 + "\n")


# ============================================================
# 10. SAVE SUMMARY AND METADATA
# ============================================================

summary_df = pd.DataFrame(all_results)

metadata_dir = os.path.join(SUBMISSION_ROOT, "metadata")
os.makedirs(metadata_dir, exist_ok=True)

summary_path = os.path.join(metadata_dir, "selected_feature_knn_summary.csv")
features_json_path = os.path.join(metadata_dir, "best_features_by_machine.json")
directions_json_path = os.path.join(metadata_dir, "score_direction_by_machine.json")

summary_df.to_csv(summary_path, index=False)

with open(features_json_path, "w") as f:
    json.dump(best_features_by_machine, f, indent=4)

with open(directions_json_path, "w") as f:
    json.dump(SCORE_DIRECTION_BY_MACHINE, f, indent=4)

print("\n" + "#" * 100)
print("FINAL SUMMARY")
print("#" * 100)

if len(summary_df) > 0:
    print(summary_df.to_string(index=False))
else:
    print("No successful machine output was created.")

print("\nSaved summary:")
print(summary_path)

print("\nSaved selected features:")
print(features_json_path)

print("\nSaved score directions:")
print(directions_json_path)


# ============================================================
# 11. VERIFY GENERATED FILES
# ============================================================

print("\n" + "#" * 100)
print("VERIFYING GENERATED FILES")
print("#" * 100)

expected_files = []

for machine in MACHINE_TYPES:
    expected_files.append(f"anomaly_score_{machine}_section_00_test.csv")
    expected_files.append(f"decision_result_{machine}_section_00_test.csv")

team_files = sorted([
    os.path.basename(f)
    for f in glob.glob(os.path.join(TEAM_DIR, "*.csv"))
])

missing = sorted(set(expected_files) - set(team_files))
extra = sorted(set(team_files) - set(expected_files))

print("Team folder:")
print(TEAM_DIR)

print("\nNumber of expected files:", len(expected_files))
print("Number of actual files  :", len(team_files))

if missing:
    print("\nMissing files:")
    for f in missing:
        print(" -", f)

if extra:
    print("\nExtra files:")
    for f in extra:
        print(" -", f)

for f in sorted(glob.glob(os.path.join(TEAM_DIR, "*.csv"))):
    df_check = pd.read_csv(f, header=None)
    print(f"{os.path.basename(f):55s} shape={df_check.shape}")

if not missing:
    print("\nAll required files were generated.")


# ============================================================
# 12. CREATE DOWNLOADABLE ZIP
# ============================================================

zip_base = os.path.join(BASE_DIR, "dcase2025_selected_feature_knn_submission")
zip_path = zip_base + ".zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(zip_base, "zip", SUBMISSION_ROOT)

print("\n" + "#" * 100)
print("ZIP CREATED")
print("#" * 100)
print(zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Automatic download is available only in Google Colab.")
    print("You can manually download the file from:")
    print(zip_path)

Output folder:
/content/selected_feature_knn_outputs

Submission team folder:
/content/dcase2025_selected_feature_knn_submission/teams/METU/TDA

PROCESSING MACHINE: AutoTrash
Feature file: /content/cubical_mel_tda_features_AutoTrash.xlsx
Selected features requested : 25
Available selected features : 25
Missing selected features   : 0
Score direction from dictionary: higher_distance_more_anomaly
GT available: NO. Metrics will not be reported.
Saved score file   : /content/selected_feature_knn_outputs/AutoTrash/anomaly_score_AutoTrash_section_00_test.csv
Saved decision file: /content/selected_feature_knn_outputs/AutoTrash/decision_result_AutoTrash_section_00_test.csv

FINISHED AutoTrash


PROCESSING MACHINE: BandSealer
Feature file: /content/cubical_mel_tda_features_BandSealer.xlsx
Selected features requested : 13
Available selected features : 13
Missing selected features   : 0
Score direction from dictionary: lower_distance_more_anomaly
GT available: NO. Metrics will not be reported.
Sa

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>